In [ ]:
# add MTBS_Event_ID field to avoid confusion downstream
import os
import arcpy

arcpy.env.overwriteOutput = True

# Define base directory
base_dir = os.path.dirname(os.path.abspath("__file__"))

# --- PATHS ---
mtbs_gdb = os.path.join(base_dir, "data", "mtbs_perimeter_data", "MTBS.gdb")
mtbs     = os.path.join(mtbs_gdb, "MTBS_1994_2024_projected_clipped")

print("Checking existing fields...")
existing_fields = [f.name for f in arcpy.ListFields(mtbs)]

# --- ADD FIELD ---
if "MTBS_Event_ID" not in existing_fields:
    print("Adding MTBS_Event_ID field...")
    arcpy.management.AddField(
        in_table=mtbs,
        field_name="MTBS_Event_ID",
        field_type="TEXT",
        field_length=50,
        field_alias="MTBS_Event_ID"
    )
    print("Field successfully added.")
else:
    print("MTBS_Event_ID field already exists. Skipping.")

In [ ]:
# Populate MTBS_Event_ID from Event_ID
import arcpy

# --- PATHS ---
mtbs_gdb = os.path.join(base_dir, "data", "mtbs_perimeter_data", "MTBS.gdb")
mtbs     = os.path.join(mtbs_gdb, "MTBS_1994_2024_projected_clipped")

print("Populating MTBS_Event_ID from Event_ID...")

# --- UPDATE CURSOR ---
# Standard MTBS schemas name the unique ID field "Event_ID"
with arcpy.da.UpdateCursor(mtbs, ["Event_ID", "MTBS_Event_ID"]) as cur:
    for eid, new in cur:
        cur.updateRow([eid, eid])

print("Successfully duplicated Event_ID to MTBS_Event_ID.")

In [ ]:
# Spatial intersect to generate SEFM event-MTBS candidate pairs
import arcpy

# --- GEODATABASE PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
mtbs_gdb        = os.path.join(base_dir, "data", "mtbs_perimeter_data", "MTBS.gdb")

# --- FEATURE CLASS PATHS ---
events = f"{classifire_gdb}\\SEFM_events_94_24"
mtbs   = f"{mtbs_gdb}\\MTBS_1994_2024_projected_clipped"
pairs  = f"{classifire_gdb}\\SEFM_MTBS_pairs"      

arcpy.env.overwriteOutput = True

print("Running spatial intersection between SEFM events and MTBS perimeters...")

# --- INTERSECT ---
arcpy.analysis.Intersect(
    in_features=[events, mtbs],
    out_feature_class=pairs,
    join_attributes="ALL"
)

# Quick quality check check
match_count = int(arcpy.management.GetCount(pairs)[0])
print(f"Intersection complete. Generated {match_count:,} overlap candidate rows in SEFM_MTBS_pairs.")

In [ ]:
# add field to mark temporal matches 
import arcpy

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
pairs          = os.path.join(classifire_gdb, "SEFM_MTBS_pairs")

print("Checking fields in SEFM_MTBS_pairs...")
existing_fields = [f.name for f in arcpy.ListFields(pairs)]

# --- ADD FIELD ---
if "temporal_match" not in existing_fields:
    print("Adding temporal_match field...")
    arcpy.management.AddField(
        in_table=pairs,
        field_name="temporal_match",
        field_type="SHORT",
        field_alias="temporal_match"
    )
    print("Field 'temporal_match' successfully added.")
else:
    print("Field 'temporal_match' already exists. Skipping.")

In [ ]:
# Look for time matches using +/- 30 days on MTBS Ig_Date
import arcpy
from datetime import datetime, timedelta

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
pairs          = os.path.join(classifire_gdb, "SEFM_MTBS_pairs")

delta = timedelta(days=30)
match_count = 0
total_count = 0

print("Evaluating temporal alignment for candidate pairs...")

# --- UPDATE CURSOR ---
fields = ["event_id", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8", "Ig_Date", "temporal_match"]

with arcpy.da.UpdateCursor(pairs, fields) as cur:
    for row in cur:
        eid, min_prebd, max_bd8, ig, tmatch = row
        total_count += 1
        
        # 1. Guard against Null or corrupt date values in your SEFM data
        if min_prebd in (None, 0) or max_bd8 in (None, 0):
            row[4] = 0
            cur.updateRow(row)
            continue
            
        # 2. Safely parse SEFM integer dates to datetime objects
        try:
            start = datetime.strptime(str(int(min_prebd)), "%Y%m%d")
            end   = datetime.strptime(str(int(max_bd8)), "%Y%m%d")
        except ValueError:
            # Handle any weirdly formatted date integers gracefully
            row[4] = 0
            cur.updateRow(row)
            continue

        # 3. Evaluate overlap window against MTBS Ignition Date
        if ig is None:
            row[4] = 0
        else:
            # MTBS buffered interval
            lo = ig - delta
            hi = ig + delta

            # Core Overlap test logic
            if (start <= hi and end >= lo):
                row[4] = 1
                match_count += 1
            else:
                row[4] = 0

        # Write the updated row back to the Geodatabase
        cur.updateRow(row)

print(f"Temporal analysis complete.")
print(f"Flagged {match_count:,} out of {total_count:,} candidate pairs as valid temporal matches.")

In [ ]:
# add mtbs_match to SEFM events
import arcpy

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
events         = os.path.join(classifire_gdb, "SEFM_events_94_24")

print("Checking fields in SEFM_events_94_24...")
existing_fields = [f.name for f in arcpy.ListFields(events)]

# --- ADD FIELD ---
if "mtbs_match" not in existing_fields:
    print("Adding mtbs_match field to master events...")
    arcpy.management.AddField(
        in_table=events,
        field_name="mtbs_match",
        field_type="TEXT",
        field_length=50,
        field_alias="mtbs_match"
    )
    print("Field 'mtbs_match' successfully added.")
else:
    print("Field 'mtbs_match' already exists. Skipping.")

In [ ]:
# build a lookup from event-MTBS Incid_Type and update master events
# If an event matches multiple Incid_Types, this checks the overlap to see which is greater (Spatial Dominance Rule for Multi-Type Matches)
import arcpy

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
pairs          = os.path.join(classifire_gdb, "SEFM_MTBS_pairs")
events         = os.path.join(classifire_gdb, "SEFM_events_94_24")

# --- 1. BUILD THE DOMINANT LOOKUP DICTIONARY ---
print("Building dominance lookup based on maximum overlap area...")

# Tracking both the winning type and the max area found so far:
# Structure: { event_id: (Incid_Type, max_area) }
event_dominance = {}

# We include Shape_Area in the cursor to evaluate the size of the intersection
fields = ["event_id", "Incid_Type", "temporal_match", "Shape_Area"]

with arcpy.da.SearchCursor(pairs, fields) as cur:
    for eid, itype, tmatch, area in cur:
        # Only process high-confidence space-time matches
        if tmatch == 1 and eid is not None and itype is not None:
            
            # If we've seen this event ID before, let the larger area win
            if eid in event_dominance:
                current_winning_type, max_area = event_dominance[eid]
                
                if area > max_area:
                    # New larger intersection found; overwrite with dominant type
                    event_dominance[eid] = (itype, area)
            else:
                # First time seeing this event ID, store it
                event_dominance[eid] = (itype, area)

# Convert our complex tracking dictionary into a clean {event_id: dominant_type} mapping
event_to_mtbs = {eid: data[0] for eid, data in event_dominance.items()}

print(f"Dominance processing complete. Resolved {len(event_to_mtbs):,} unique master events.")

# --- 2. UPDATE THE MASTER EVENTS LAYER ---
print("Writing dominant MTBS labels to master SEFM_events_94_24 layer...")
updated_count = 0

with arcpy.da.UpdateCursor(events, ["event_id", "mtbs_match"]) as cur:
    for row in cur:
        eid = row[0]
        
        if eid in event_to_mtbs:
            row[1] = event_to_mtbs[eid]
            updated_count += 1
        else:
            row[1] = None  # Keeps unmatched fields as a clean database NULL
            
        cur.updateRow(row)

print(f"Assigned dominant MTBS labels to {updated_count:,} master events.")

In [ ]:
# add sefm_match to mtbs
import arcpy

# --- PATHS ---
mtbs_gdb = os.path.join(base_dir, "data", "mtbs_perimeter_data", "MTBS.gdb")
mtbs     = os.path.join(mtbs_gdb, "MTBS_1994_2024_projected_clipped")

print("Checking fields in MTBS perimeters...")
existing_fields = [f.name for f in arcpy.ListFields(mtbs)]

# --- ADD FIELD ---
if "sefm_match" not in existing_fields:
    print("Adding sefm_match field to MTBS layer...")
    arcpy.management.AddField(
        in_table=mtbs,
        field_name="sefm_match",
        field_type="SHORT",
        field_alias="sefm_match"
    )
    print("Field 'sefm_match' successfully added.")
else:
    print("Field 'sefm_match' already exists. Skipping.")

In [ ]:
# build mtbs-matched lookup and update MTBS perimeters
import arcpy

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
mtbs_gdb       = os.path.join(base_dir, "data", "mtbs_perimeter_data", "MTBS.gdb")

pairs = f"{classifire_gdb}\\SEFM_MTBS_pairs"
mtbs  = f"{mtbs_gdb}\\MTBS_1994_2024_projected_clipped"

# --- 1. BUILD THE MATCHED LOOKUP SET ---
print("Scanning pairs for valid MTBS fires...")
mtbs_ids = set()

with arcpy.da.SearchCursor(pairs, ["MTBS_Event_ID", "temporal_match"]) as cur:
    for fid, tmatch in cur:
        if tmatch == 1 and fid is not None:
            mtbs_ids.add(fid)

print(f"Lookup set complete. Found {len(mtbs_ids):,} unique MTBS fires captured by SEFM.")

# --- 2. UPDATE THE MTBS PERIMETERS LAYER ---
print("Writing validation flags to MTBS perimeters...")
updated_count = 0

with arcpy.da.UpdateCursor(mtbs, ["MTBS_Event_ID", "sefm_match"]) as cur:
    for row in cur:
        fid = row[0]
        
        if fid in mtbs_ids:
            row[1] = 1
            updated_count += 1
        else:
            row[1] = None  # Keeps unmatched agency fires as clean database NULLs
            
        cur.updateRow(row)

print(f"Success! Flagged {updated_count:,} MTBS fires as 'Detected' by SEFM.")
print("All remaining unmapped agency fires have been left as NULL.")

In [ ]:
# ===================================================================
# Create a new true DATE field in SEFM events and populate it
#          with the companion MTBS Ig_Date for valid temporal matches.
# ===================================================================
import arcpy
import os

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
pairs          = os.path.join(classifire_gdb, "SEFM_MTBS_pairs")
events         = os.path.join(classifire_gdb, "SEFM_events_94_24")

target_field = "mtbs_ig_date"

# --- STEP 1: INITIALIZE THE NEW DATE FIELD ---
print(f"Checking fields in {os.path.basename(events)}...")
existing_fields = [f.name for f in arcpy.ListFields(events)]

if target_field not in existing_fields:
    print(f"Adding true DATE field '{target_field}' to master SEFM events...")
    arcpy.management.AddField(
        in_table=events,
        field_name=target_field,
        field_type="DATE",
        field_alias="mtbs_ig_date"
    )
    print(f"Field '{target_field}' successfully added.")
else:
    print(f"Field '{target_field}' already exists. Skipping initialization.")


# --- STEP 2: BUILD LOOKUP DICTIONARY FROM PAIRS ---
print("Building date mapping dictionary from valid temporal pairs...")
date_lookup = {}

# We pull event_id and Ig_Date. 
# NOTE: If MTBS date field uses a different capitalization (like ig_date), adjust below.
with arcpy.da.SearchCursor(pairs, ["event_id", "Ig_Date", "temporal_match"]) as search_cur:
    for eid, ig_date, tmatch in search_cur:
        if tmatch == 1 and eid is not None and ig_date is not None:
            # Map the SEFM event ID string to the date object
            date_lookup[str(eid)] = ig_date

print(f"Mapped ignition dates for {len(date_lookup):,} matched event IDs.")


# --- STEP 3: UPDATE THE MASTER SEFM LAYER ---
print(f"Writing MTBS ignition dates to master SEFM layer...")
updated_count = 0

with arcpy.da.UpdateCursor(events, ["event_id", target_field]) as update_cur:
    for row in update_cur:
        sefm_id = str(row[0]) if row[0] is not None else None
        
        if sefm_id in date_lookup:
            row[1] = date_lookup[sefm_id]
            updated_count += 1
        else:
            row[1] = None  # Leave unmatched events as a clean database NULL date
            
        update_cur.updateRow(row)

# Clear schema locks
arcpy.management.ClearWorkspaceCache(classifire_gdb)

print("-" * 60)
print("SUCCESS: Master SEFM layer updated with MTBS ignition dates.")
print(f"Populated dates for {updated_count:,} matching features.")
print("-" * 60)